In [ ]:
!pip install dash==2.14.2 pyngrok
!pip uninstall -y dash
!pip install dash==2.14.2 flask==2.2.5 werkzeug==2.2.3
!pip install dash-bootstrap-components
!pip install pandas numpy matplotlib seaborn plotly dash mlxtend scikit-learn statsmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.8
    Uninstalling Werkzeug-3.1.8:
      Successfully uninstalled Werkzeug-3.1.8
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.3
    Uninstalling Flask-3.1.3:
      Successfully uninstalled Flask-3.1.3
Found existing installation: dash 2.14.2
Uninstalling dash-2.14.2:
  Successfully uninstalled dash-2.14.2
  Using cached dash-2.14.2-py3-none-any.whl.metadata (11 kB)
Using cached dash-2.14.2-py3-none-any.whl (10.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeu

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pour le frequent pattern mining
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Pour la prévision temporelle
from sklearn.linear_model import LinearRegression

# Pour le clustering spatial
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print(" Toutes les bibliothèques sont importées.")

 Toutes les bibliothèques sont importées.


In [ ]:
import pandas as pd

# ============================================================
# Fonction : load_data
# Input  : chemin vers le fichier CSV (string)
# Output : DataFrame pandas
# ============================================================
def load_data(file_path):
    """
    Charge le fichier CSV dans un DataFrame pandas.
    Vérifie que le fichier existe et affiche un aperçu.
    """
    df = pd.read_csv(file_path)
    print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    return df

df = load_data("/content/earthquake_data_tsunami (1).csv")

Données chargées : 782 lignes, 13 colonnes


In [ ]:
# ============================================================
# Fonction : explore_data
# Input  : df (DataFrame)
# Output : affichage des principales informations du dataset
# ============================================================

def explore_data(df):
    """
    Explore le dataset et affiche des informations utiles
    pour mieux comprendre les données.
    """

    # Affiche une ligne de séparation pour améliorer la lisibilité
    print("=" * 60)

    # Affiche le titre de la section
    print("EXPLORATION DES DONNÉES")

    # Affiche une deuxième ligne de séparation
    print("=" * 60)

    # df.shape renvoie un tuple (nombre_lignes, nombre_colonnes)
    # shape[0] = nombre de lignes
    # shape[1] = nombre de colonnes
    print(f"\nDimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")

    # Affiche le type de données de chaque colonne
    # int64 : entier
    # float64 : nombre décimal
    # object : texte
    # datetime : date
    print("\nTypes de données par colonne :")
    print(df.dtypes)

    # Détecte les valeurs manquantes (NaN)
    # isnull() retourne True pour les cellules vides
    # sum() compte le nombre de True par colonne
    print("\nValeurs manquantes par colonne :")
    missing = df.isnull().sum()
    print(missing)

    # Génère les statistiques descriptives des colonnes numériques :
    # count : nombre de valeurs
    # mean  : moyenne
    # std   : écart-type
    # min   : valeur minimale
    # 25%   : premier quartile
    # 50%   : médiane
    # 75%   : troisième quartile
    # max   : valeur maximale
    print("\nStatistiques descriptives :")
    print(df.describe().round(2))

    # Compte le nombre d'occurrences de chaque valeur
    # dans la colonne tsunami
    # Exemple :
    # 0 = pas de tsunami
    # 1 = tsunami
    print("\nDistribution de la variable 'tsunami' :")
    print(df['tsunami'].value_counts())

    # La moyenne d'une variable binaire (0/1)
    # correspond directement à la proportion de 1
    # Multiplication par 100 pour obtenir un pourcentage
    print(f"\n→ {df['tsunami'].mean()*100:.1f}% des séismes ont généré un tsunami")

    # Recherche l'année la plus ancienne et la plus récente
    # du dataset afin de connaître la période couverte
    print(f"\nPériode couverte : {df['Year'].min()} à {df['Year'].max()}")

In [ ]:
explore_data(df)

EXPLORATION DES DONNÉES

Dimensions : 782 lignes × 13 colonnes

Types de données par colonne :
magnitude    float64
cdi            int64
mmi            int64
sig            int64
nst            int64
dmin         float64
gap          float64
depth        float64
latitude     float64
longitude    float64
Year           int64
Month          int64
tsunami        int64
dtype: object

Valeurs manquantes par colonne :
magnitude    0
cdi          0
mmi          0
sig          0
nst          0
dmin         0
gap          0
depth        0
latitude     0
longitude    0
Year         0
Month        0
tsunami      0
dtype: int64

Statistiques descriptives :
       magnitude     cdi     mmi      sig     nst    dmin     gap   depth  \
count     782.00  782.00  782.00   782.00  782.00  782.00  782.00  782.00   
mean        6.94    4.33    5.96   870.11  230.25    1.33   25.04   75.88   
std         0.45    3.17    1.46   322.47  250.19    2.22   24.23  137.28   
min         6.50    0.00    1.00   650.

**INTERPRETATION RESULTATS**
1. Taille du jeu de données

Le dataset contient 782 séismes décrits par 13 variables.

2. Qualité des données

Toutes les colonnes ont 0 valeur manquante.

- Les données sont complètes et peuvent être analysées sans nettoyage supplémentaire.

3. Magnitude des séismes
Magnitude moyenne : 6,94
Magnitude minimale : 6,5
Magnitude maximale : 9,1

- Le dataset contient uniquement des séismes forts à très forts.
- Le plus puissant a atteint une magnitude de 9,1, ce qui correspond à un événement majeur.

4. Profondeur des séismes
Profondeur moyenne : 75,88 km
Profondeur minimale : 2,7 km
Profondeur maximale : 670,81 km

- Certains séismes sont très superficiels tandis que d'autres se produisent très profondément sous la Terre.

5. Répartition géographique
Latitude : de -61,85° à 71,63°
Longitude : de -179,97° à 179,66°

- Les séismes sont répartis dans de nombreuses régions du monde.

6. Période étudiée
De 2001 à 2022

- L'analyse couvre 22 années d'activité sismique.

7. Tsunamis

Répartition :

Sans tsunami : 478 séismes
Avec tsunami : 304 séismes

- Environ 38,9 % des séismes ont provoqué un tsunami.
- La majorité des séismes (61,1 %) n'ont pas généré de tsunami.

## Indicateur 1  Magnitude maximale par année

**Ce que ça représente** : Pour chaque année entre 2001 et 2022, on cherche le séisme le plus puissant.
Cela permet de voir si certaines années ont été particulièrement actives sismiquement.

**Comment c'est calculé** : On regroupe les données par `Year` et on prend le `max` de `magnitude`.

**Comment l'interpréter** : Une valeur élevée (ex: 9.1 en 2011) indique une année avec un séisme majeur.

In [ ]:
# ============================================================
# Fonction : query_max_magnitude_per_year
# Entrée  : df (DataFrame contenant les données des séismes)
# Sortie  : DataFrame regroupé + graphique interactif
# ============================================================

def query_max_magnitude_per_year(df):
    """
    Cette fonction analyse les séismes année par année.

    Son objectif est de trouver, pour chaque année,
    le séisme ayant la magnitude la plus élevée.

    Elle affiche :
    - un tableau récapitulatif des magnitudes maximales
    - un graphique montrant l'évolution des magnitudes
      maximales au fil des années

    Paramètre :
        df : DataFrame contenant les données des séismes

    Retour :
        result : DataFrame contenant l'année et la magnitude maximale
    """

    # --------------------------------------------------------
    # ÉTAPE 1 : Regrouper les données par année
    # --------------------------------------------------------
    # groupby('Year')
    #
    # Cette instruction rassemble toutes les lignes
    # appartenant à la même année.
    #
    # Exemple :
    #
    # Year    Magnitude
    # 2001      5.2
    # 2001      6.8
    # 2001      7.1
    #
    # Toutes les lignes de l'année 2001
    # sont regroupées dans un même groupe.
    # --------------------------------------------------------

    # --------------------------------------------------------
    # ['magnitude']
    #
    # On sélectionne uniquement la colonne magnitude
    # car c'est la variable que l'on souhaite analyser.
    # --------------------------------------------------------

    # --------------------------------------------------------
    # .max()
    #
    # Pour chaque année, on calcule la plus grande magnitude.
    #
    # Exemple :
    #
    # 2001 → max(5.2, 6.8, 7.1) = 7.1
    # 2002 → max(4.5, 6.3, 5.8) = 6.3
    #
    # On obtient donc la magnitude maximale
    # enregistrée pour chaque année.
    # --------------------------------------------------------

    # --------------------------------------------------------
    # .reset_index()
    #
    # Après un groupby, l'année devient un index.
    #
    # Cette instruction remet l'année sous forme
    # de colonne classique afin de faciliter
    # les affichages et la création des graphiques.
    # --------------------------------------------------------

    result = df.groupby('Year')['magnitude'].max().reset_index()

    # --------------------------------------------------------
    # ÉTAPE 2 : Renommer les colonnes
    # --------------------------------------------------------
    # On attribue des noms plus explicites
    # pour rendre le tableau plus lisible.
    #
    # Year → Année
    # Max_Magnitude → Magnitude maximale
    # --------------------------------------------------------

    result.columns = ['Year', 'Max_Magnitude']

    # --------------------------------------------------------
    # ÉTAPE 3 : Affichage du tableau
    # --------------------------------------------------------
    # Affiche pour chaque année
    # la magnitude maximale observée.
    # --------------------------------------------------------

    print(" Magnitude maximale par année :")

    print(result.to_string(index=False))

    # --------------------------------------------------------
    # ÉTAPE 4 : Création du graphique
    # --------------------------------------------------------
    # px.bar() permet de créer un diagramme en barres.
    #
    # Chaque barre représente une année.
    #
    # La hauteur de la barre correspond
    # à la magnitude maximale observée cette année-là.
    # --------------------------------------------------------

    fig = px.bar(

        # DataFrame utilisé pour construire le graphique
        result,

        # Axe horizontal : les années
        x='Year',

        # Axe vertical : magnitude maximale
        y='Max_Magnitude',

        # Titre affiché en haut du graphique
        title='Magnitude maximale des séismes par année (2001–2022)',

        # Noms affichés sur les axes
        labels={
            'Max_Magnitude': 'Magnitude max',
            'Year': 'Année'
        },

        # Couleur des barres basée sur la magnitude
        color='Max_Magnitude',

        # Palette de couleurs rouges
        # Les séismes les plus puissants apparaissent
        # avec une couleur plus foncée.
        color_continuous_scale='Reds'
    )

    # --------------------------------------------------------
    # ÉTAPE 5 : Mise en forme du graphique
    # --------------------------------------------------------
    # La légende n'est pas utile ici
    # car la couleur représente déjà
    # directement la magnitude.
    # --------------------------------------------------------

    fig.update_layout(showlegend=False)

    # --------------------------------------------------------
    # ÉTAPE 6 : Affichage du graphique
    # --------------------------------------------------------
    # Ouvre un graphique interactif Plotly.
    #
    # L'utilisateur peut :
    # - zoomer
    # - déplacer le graphique
    # - passer la souris sur une barre
    #   pour voir les valeurs exactes
    # --------------------------------------------------------

    fig.show()

    # --------------------------------------------------------
    # ÉTAPE 7 : Retour du résultat
    # --------------------------------------------------------
    # On retourne le DataFrame final.
    #
    # Cela permet de réutiliser les résultats
    # dans d'autres analyses ou graphiques.
    # --------------------------------------------------------

    return result

In [ ]:
import plotly.express as px

result = query_max_magnitude_per_year(df)

result.head()

 Magnitude maximale par année :
 Year  Max_Magnitude
 2001           8.40
 2002           7.90
 2003           8.16
 2004           9.10
 2005           8.60
 2006           8.30
 2007           8.40
 2008           7.90
 2009           8.10
 2010           8.80
 2011           9.10
 2012           8.60
 2013           8.30
 2014           8.20
 2015           8.30
 2016           7.90
 2017           8.20
 2018           8.20
 2019           8.00
 2020           7.80
 2021           8.20
 2022           7.60


,Year,Max_Magnitude
0,2001,8.40
1,2002,7.90
2,2003,8.16
3,2004,9.10
4,2005,8.60


Cette fonction réalise une opération de groupement avec groupby(). Elle regroupe tous les séismes par année puis calcule, grâce à max(), la magnitude la plus élevée observée chaque année. Les résultats sont stockés dans un nouveau DataFrame nommé result. Ensuite, un graphique en barres est généré avec Plotly afin de visualiser facilement les années ayant connu les séismes les plus puissants. Enfin, la fonction retourne le tableau obtenu pour permettre d'autres analyses.

INTERPRETATION

Ce tableau présente le séisme le plus puissant de chaque année entre 2001 et 2022.

On observe que les magnitudes maximales restent toujours élevées, entre 7,6 et 9,1, ce qui montre une forte activité sismique.

Les années 2004 et 2011 enregistrent les séismes les plus puissants avec une magnitude de 9,1. À l'inverse, 2022 présente la valeur la plus faible (7,6), mais il s'agit toujours d'un séisme important.

Globalement, les valeurs varient d'une année à l'autre sans augmentation ni diminution nette.

## Indicateur 2 :  Règles d'association entre caractéristiques des séismes

**Ce que ça représente** : On cherche des combinaisons fréquentes de caractéristiques qui apparaissent
souvent ensemble. Par exemple : "les séismes profonds ET de forte magnitude génèrent-ils souvent des tsunamis ?"

**Comment c'est calculé** : On discrétise les variables numériques en catégories (ex: magnitude "haute/basse"),
puis on applique l'algorithme Apriori pour trouver les associations fréquentes.

**Comment l'interpréter** : Une règle avec "confiance = 0.85" et "lift > 1" signifie que la combinaison
de conditions prédit souvent le résultat (tsunami = oui).

In [ ]:
# ============================================================
# Fonction : query_frequent_patterns
# Entrée  : df (DataFrame contenant les données des séismes)
# Sortie  : règles d'association + graphique interactif
# ============================================================

def query_frequent_patterns(df):

    """
    Cette fonction cherche des relations fréquentes entre
    différentes caractéristiques des séismes.

    Exemple :
    "Les séismes de forte magnitude provoquent-ils souvent un tsunami ?"

    Pour cela, elle utilise l'algorithme Apriori qui permet
    de découvrir des associations récurrentes dans les données.
    """

    # --------------------------------------------------------
    # Étape 1 : Création du DataFrame binaire
    # --------------------------------------------------------

    # Création d'un nouveau DataFrame vide qui contiendra
    # uniquement des variables binaires (0 ou 1)
    df_bin = pd.DataFrame()

    # 1 si la magnitude est supérieure ou égale à 7.0, sinon 0
    df_bin['high_magnitude'] = (df['magnitude'] >= 7.0).astype(int)

    # 1 si la profondeur est inférieure à 30 km, sinon 0
    df_bin['low_depth'] = (df['depth'] < 30).astype(int)

    # 1 si l'indice de significativité dépasse 700, sinon 0
    df_bin['high_sig'] = (df['sig'] > 700).astype(int)

    # 1 si l'intensité ressentie (CDI) est supérieure à 5, sinon 0
    df_bin['high_cdi'] = (df['cdi'] > 5).astype(int)

    # Copie directe de la variable tsunami (0 = non, 1 = oui)
    df_bin['tsunami'] = df['tsunami'].astype(int)

    # --------------------------------------------------------
    # Étape 2 : Conversion en booléens
    # --------------------------------------------------------

    # Apriori travaille avec des valeurs True/False
    # Conversion des 0/1 en booléens
    df_bool = df_bin.astype(bool)

    # --------------------------------------------------------
    # Étape 3 : Recherche des itemsets fréquents
    # --------------------------------------------------------

    # Application de l'algorithme Apriori
    # min_support=0.1 signifie que l'association doit apparaître
    # dans au moins 10% des observations
    # use_colnames=True permet d'afficher les noms des variables
    frequent_items = apriori(
        df_bool,
        min_support=0.1,
        use_colnames=True
    )

    # Affichage des itemsets fréquents trouvés
    print("\n" + "="*60)
    print("ITEMSETS FREQUENTS TROUVES PAR APRIORI")
    print("="*60)
    print(frequent_items)

    # --------------------------------------------------------
    # Étape 4 : Génération des règles d'association
    # --------------------------------------------------------

    # Création des règles d'association à partir des itemsets
    # confidence >= 0.5 signifie qu'une règle doit être vraie
    # au moins 50% du temps pour être conservée
    rules = association_rules(
        frequent_items,
        metric='confidence',
        min_threshold=0.5
    )

    # Vérification si aucune règle n'a été trouvée
    if rules.empty:
        print("\nAucune règle d'association trouvée.")
        return rules

    # --------------------------------------------------------
    # Étape 5 : Tri des règles par lift
    # --------------------------------------------------------

    # Classement des règles selon le lift
    # Les règles les plus intéressantes apparaissent en premier
    rules = rules.sort_values(
        by='lift',
        ascending=False
    )

    # --------------------------------------------------------
    # Étape 6 : Conversion des frozensets en texte
    # --------------------------------------------------------
    # Nécessaire pour Plotly
    # --------------------------------------------------------

    # Transformation des antécédents en chaîne de caractères
    # Exemple :
    # {'high_magnitude', 'high_sig'}
    # devient
    # "high_magnitude, high_sig"
    rules['antecedents_str'] = rules['antecedents'].apply(
        lambda x: ', '.join(list(x))
    )

    # Même traitement pour les conséquences
    rules['consequents_str'] = rules['consequents'].apply(
        lambda x: ', '.join(list(x))
    )

    # --------------------------------------------------------
    # Étape 7 : Affichage des meilleures règles
    # --------------------------------------------------------

    print("\n" + "="*60)
    print("TOP 10 REGLES D'ASSOCIATION")
    print("="*60)

    # Affichage des 10 règles les plus intéressantes
    # avec leurs principaux indicateurs
    print(
        rules[
            [
                'antecedents_str',  # condition de départ
                'consequents_str',  # conséquence observée
                'support',          # fréquence d'apparition
                'confidence',       # fiabilité de la règle
                'lift'              # force de l'association
            ]
        ]
        .head(10)
        .to_string(index=False)
    )

    # Affichage du nombre total de règles trouvées
    print(f"\nNombre total de règles trouvées : {len(rules)}")

    # --------------------------------------------------------
    # Étape 8 : Graphique interactif
    # --------------------------------------------------------

    # Création d'un nuage de points interactif
    # Chaque point représente une règle d'association
    fig = px.scatter(
        rules,

        # Axe horizontal : support
        x='support',

        # Axe vertical : confidence
        y='confidence',

        # Taille des bulles : lift
        size='lift',

        # Couleur des bulles : lift
        color='lift',

        # Informations affichées au survol
        hover_data=[
            'antecedents_str',
            'consequents_str'
        ],

        # Titre du graphique
        title="Règles d'association - Support vs Confidence (taille = Lift)",

        # Palette de couleurs
        color_continuous_scale='Viridis'
    )

    # Affichage du graphique interactif
    fig.show()

    # --------------------------------------------------------
    # Étape 9 : Retour des résultats
    # --------------------------------------------------------

    # Retourne le DataFrame contenant toutes les règles
    # pour une éventuelle utilisation ultérieure
    return rules

In [ ]:
rules = query_frequent_patterns(df)

display(rules.head())


ITEMSETS FREQUENTS TROUVES PAR APRIORI
     support                               itemsets
0   0.361893                       (high_magnitude)
1   0.551151                            (low_depth)
2   0.696931                             (high_sig)
3   0.416880                             (high_cdi)
4   0.388747                              (tsunami)
5   0.172634            (high_magnitude, low_depth)
6   0.361893             (high_magnitude, high_sig)
7   0.195652             (high_magnitude, high_cdi)
8   0.140665              (high_magnitude, tsunami)
9   0.372123                  (low_depth, high_sig)
10  0.248082                  (low_depth, high_cdi)
11  0.208440                   (low_depth, tsunami)
12  0.361893                   (high_sig, high_cdi)
13  0.272379                    (high_sig, tsunami)
14  0.196931                    (tsunami, high_cdi)
15  0.172634  (high_magnitude, low_depth, high_sig)
16  0.195652   (high_magnitude, high_sig, high_cdi)
17  0.140665    (high_ma

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_str,consequents_str
14,"(high_sig, high_cdi)",(high_magnitude),0.361893,0.361893,0.195652,0.540636,1.493913,1.0,0.064686,1.389111,0.518121,0.370460,0.280115,0.540636,"high_sig, high_cdi",high_magnitude
15,(high_magnitude),"(high_sig, high_cdi)",0.361893,0.361893,0.195652,0.540636,1.493913,1.0,0.064686,1.389111,0.518121,0.370460,0.280115,0.540636,high_magnitude,"high_sig, high_cdi"
24,"(high_sig, tsunami)",(high_cdi),0.272379,0.416880,0.167519,0.615023,1.475302,1.0,0.053970,1.514690,0.442775,0.321078,0.339799,0.508432,"high_sig, tsunami",high_cdi
11,"(high_magnitude, low_depth)",(high_sig),0.172634,0.696931,0.172634,1.000000,1.434862,1.0,0.052320,inf,0.366306,0.247706,1.000000,0.623853,"high_magnitude, low_depth",high_sig
13,"(high_magnitude, high_cdi)",(high_sig),0.195652,0.696931,0.195652,1.000000,1.434862,1.0,0.059296,inf,0.376789,0.280734,1.000000,0.640367,"high_magnitude, high_cdi",high_sig


INTERPRETATION


1.  ITEMSETS FREQUENTS

*   high_sig (69%) → beaucoup de séismes sont “importants”

*   low_depth (55%) → beaucoup de séismes sont superficiels

*   high_cdi (41%) → beaucoup ont été fortement ressentis

*   tsunami (38%) → les tsunamis restent assez fréquents


2.   REGLES D'ASSOCIATIONS

Règle  1 :  

high_magnitude → high_sig + high_cdi confidence ≈ 0.54 lift ≈ 1.49

--> Quand un séisme est de forte magnitude, il est souvent significatif et fortement ressenti.


Règle  2 :  

high_sig → high_magnitude confidence ≈ 0.52 lift ≈ 1.43

--> Les séismes significatifs sont souvent de forte magnitude.

Règle 3:

low_depth + high_magnitude → high_sig confidence = 1.00

--> Tous les séismes qui sont à la fois superficiels et forts sont aussi significatifs.

## **évolution temporelle + modèle de régression**

In [ ]:
# ============================================================
# Fonction : query_temporal_trend
# Entrée  : df (DataFrame contenant les données des séismes)
# Sortie  : évolution temporelle + modèle de régression
# ============================================================

def query_temporal_trend(df):
    """
    Cette fonction analyse l'évolution du nombre de séismes
    au fil des années.

    Elle permet aussi de prévoir les années futures
    grâce à une régression linéaire.
    """

    # --------------------------------------------------------
    # ÉTAPE 1 : Compter le nombre de séismes par année
    # --------------------------------------------------------

    # Regrouper toutes les lignes par année
    # puis compter combien de séismes existent chaque année
    # Le résultat est stocké dans un DataFrame appelé yearly
    yearly = df.groupby('Year').size().reset_index(name='count')

    # --------------------------------------------------------
    # ÉTAPE 2 : Préparer les données pour la régression
    # --------------------------------------------------------

    # X contient les années (variable indépendante)
    X = yearly[['Year']]

    # y contient le nombre de séismes observés
    # c'est la variable que le modèle devra prédire
    y = yearly['count']

    # --------------------------------------------------------
    # ÉTAPE 3 : Création du modèle de régression linéaire
    # --------------------------------------------------------

    # Création d'un modèle de Machine Learning
    # de type régression linéaire
    model = LinearRegression()

    # Entraînement du modèle à partir des données historiques
    model.fit(X, y)

    # --------------------------------------------------------
    # ÉTAPE 4 : Prévisions pour les années futures
    # --------------------------------------------------------

    # Création d'un DataFrame contenant les années
    # pour lesquelles on souhaite faire des prédictions
    future_years = pd.DataFrame({'Year': [2023, 2024, 2025]})

    # Le modèle estime le nombre de séismes
    # pour chacune de ces années
    predictions = model.predict(future_years)

    # --------------------------------------------------------
    # ÉTAPE 5 : Affichage des résultats numériques
    # --------------------------------------------------------

    # Afficher le tableau du nombre de séismes par année
    print(" Nombre de séismes par année :")

    # Affichage complet du DataFrame sans index
    print(yearly.to_string(index=False))

    # Affichage de la pente de la droite de régression
    # Une valeur positive signifie une augmentation
    # Une valeur négative signifie une diminution
    print(f"\n Tendance globale : +{model.coef_[0]:.1f} séismes par an")

    # Afficher les prévisions calculées par le modèle
    print(f" Prévisions :")

    # Prévision pour l'année 2023
    print(f"2023 → {predictions[0]:.0f} séismes")

    # Prévision pour l'année 2024
    print(f"2024 → {predictions[1]:.0f} séismes")

    # Prévision pour l'année 2025
    print(f"2025 → {predictions[2]:.0f} séismes")

    # --------------------------------------------------------
    # ÉTAPE 6 : Création du graphique
    # --------------------------------------------------------

    # Création d'une figure vide Plotly
    fig = go.Figure()

    # Ajout d'un graphique en barres représentant
    # les données réelles observées dans le dataset
    fig.add_trace(go.Bar(

        # Axe X : années
        x=yearly['Year'],

        # Axe Y : nombre de séismes
        y=yearly['count'],

        # Nom affiché dans la légende
        name='Séismes réels',

        # Couleur des barres
        marker_color='steelblue'
    ))

    # Calcul des valeurs de la droite de tendance
    # pour toutes les années historiques
    trend_y = model.predict(X)

    # Ajout de la ligne de tendance
    fig.add_trace(go.Scatter(

        # Années
        x=yearly['Year'],

        # Valeurs prédites par la régression
        y=trend_y,

        # Affichage sous forme de ligne
        mode='lines',

        # Nom dans la légende
        name='Tendance',

        # Couleur et épaisseur de la ligne
        line=dict(color='red', width=2)
    ))

    # Ajout des prévisions futures
    fig.add_trace(go.Scatter(

        # Années futures
        x=future_years['Year'],

        # Valeurs prédites
        y=predictions,

        # Affichage sous forme de points reliés
        mode='markers+lines',

        # Nom dans la légende
        name='Prévisions',

        # Ligne orange en pointillés
        line=dict(color='orange', dash='dash'),

        # Taille des points
        marker=dict(size=10)
    ))

    # --------------------------------------------------------
    # ÉTAPE 7 : Mise en forme du graphique
    # --------------------------------------------------------

    # Ajout du titre et des noms des axes
    fig.update_layout(

        # Titre principal du graphique
        title='Évolution temporelle du nombre de séismes (2001–2022) + prévisions',

        # Nom de l'axe horizontal
        xaxis_title='Année',

        # Nom de l'axe vertical
        yaxis_title='Nombre de séismes'
    )

    # Affichage du graphique interactif
    fig.show()

    # --------------------------------------------------------
    # ÉTAPE 8 : Retour des résultats
    # --------------------------------------------------------

    # Retourne :
    # - yearly : tableau des statistiques annuelles
    # - model : modèle entraîné pouvant être réutilisé
    return yearly, model

In [ ]:
yearly, model = query_temporal_trend(df)

 Nombre de séismes par année :
 Year  count
 2001     28
 2002     25
 2003     31
 2004     32
 2005     28
 2006     26
 2007     37
 2008     25
 2009     26
 2010     41
 2011     34
 2012     31
 2013     53
 2014     48
 2015     53
 2016     43
 2017     36
 2018     43
 2019     33
 2020     27
 2021     42
 2022     40

 Tendance globale : +0.7 séismes par an
 Prévisions :
2023 → 43 séismes
2024 → 44 séismes
2025 → 45 séismes


INTERPREATATION


1. Évolution du nombre de séismes entre 2001 et 2022

L'analyse montre que le nombre de séismes varie d'une année à l'autre, mais présente une tendance générale à la hausse.

En 2001, on observe 28 séismes.
En 2022, on en observe 40 séismes.
Les années les plus actives sont :
2013 : 53 séismes
2015 : 53 séismes
L'année la moins active est :
2002 : 25 séismes (également 2008 avec 25 séismes).


2. Analyse de la tendance

Le modèle de régression linéaire a calculé une pente de : +0,7 séisme par an

Cela signifie qu'en moyenne, le nombre de séismes augmente d'environ : 0,7 événement chaque année.

Autrement dit, tous les 10 ans, on observe approximativement : 7 séismes supplémentaires.

La ligne rouge sur le graphique représente cette tendance générale croissante.

3. Interprétation du graphique

Le graphique contient trois éléments :

Barres bleues : données réelles

Elles représentent le nombre réel de séismes enregistrés chaque année.

On observe notamment :

une hausse importante entre 2010 et 2015 ;
un pic d'activité en 2013 et 2015 ;
une baisse temporaire en 2020 ;
une remontée en 2021 et 2022.
Ligne rouge : tendance

La ligne rouge correspond à la régression linéaire.

Même si certaines années sont plus faibles ou plus fortes que d'autres, la ligne monte progressivement, ce qui confirme une augmentation globale du nombre de séismes au cours de la période étudiée.

Points orange : prévisions

Ils représentent les estimations du modèle pour les années futures.

Le modèle prévoit environ :

43 séismes en 2023
44 séismes en 2024
45 séismes en 2025

In [ ]:
# ============================================================
# Fonction : query_spatial_clustering
# Entrée  : df (DataFrame contenant les séismes)
# Sortie  : DataFrame avec clusters + carte interactive
# ============================================================

def query_spatial_clustering(df, n_clusters=6):
    """
    Cette fonction regroupe les séismes par zones géographiques
    en utilisant l'algorithme KMeans.

    Objectif :
    Identifier les zones sismiques les plus actives dans le monde.
    """

    # --------------------------------------------------------
    # ÉTAPE 1 : Sélection des coordonnées géographiques
    # --------------------------------------------------------

    # Récupération des colonnes latitude et longitude
    # car elles permettent de localiser les séismes
    coords = df[['latitude', 'longitude']].copy()

    # --------------------------------------------------------
    # ÉTAPE 2 : Normalisation des données
    # --------------------------------------------------------

    # Création d'un objet de normalisation
    scaler = StandardScaler()

    # Mise à l'échelle des coordonnées
    # pour que latitude et longitude aient le même poids
    coords_scaled = scaler.fit_transform(coords)

    # --------------------------------------------------------
    # ÉTAPE 3 : Application de KMeans
    # --------------------------------------------------------

    # Création du modèle KMeans
    # n_clusters = nombre de groupes à créer
    # random_state = garantit les mêmes résultats à chaque exécution
    # n_init = nombre de tentatives pour trouver la meilleure solution
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

    # Copie du DataFrame pour conserver les données originales
    df = df.copy()

    # Attribution d'un cluster à chaque séisme
    # KMeans analyse les coordonnées et assigne un groupe
    df['cluster'] = kmeans.fit_predict(coords_scaled)

    # Conversion du numéro du cluster en texte
    # pour faciliter l'affichage des couleurs sur la carte
    df['cluster'] = df['cluster'].astype(str)

    # --------------------------------------------------------
    # ÉTAPE 4 : Analyse des clusters
    # --------------------------------------------------------

    # Calcul de statistiques pour chaque cluster
    cluster_stats = df.groupby('cluster').agg(

        # Nombre de séismes dans le cluster
        nb_seismes=('magnitude', 'count'),

        # Magnitude moyenne des séismes du cluster
        magnitude_moy=('magnitude', 'mean'),

        # Moyenne de la variable tsunami
        # proche de 1 = beaucoup de tsunamis
        # proche de 0 = peu de tsunamis
        taux_tsunami=('tsunami', 'mean')

    ).round(3)  # arrondit les résultats à 3 décimales

    # Affichage des statistiques calculées
    print("Statistiques par cluster spatial :")

    # Affichage du tableau complet
    print(cluster_stats.to_string())

    # --------------------------------------------------------
    # ÉTAPE 5 : Visualisation sur une carte
    # --------------------------------------------------------

    # Création d'une carte interactive mondiale
    fig = px.scatter_geo(

        # Données à afficher
        df,

        # Latitude des séismes
        lat='latitude',

        # Longitude des séismes
        lon='longitude',

        # Une couleur différente pour chaque cluster
        color='cluster',

        # Taille du point proportionnelle à la magnitude
        size='magnitude',

        # Informations affichées au survol de la souris
        hover_data=['Year', 'magnitude', 'depth', 'tsunami'],

        # Titre de la carte
        title=f'Clustering spatial des séismes ({n_clusters} zones)',

        # Type de projection géographique
        projection='natural earth'
    )

    # Affichage de la carte interactive
    fig.show()

    # --------------------------------------------------------
    # ÉTAPE 6 : Retour des résultats
    # --------------------------------------------------------

    # Retourne :
    # df -> données avec le cluster attribué à chaque séisme
    # cluster_stats -> statistiques calculées pour chaque cluster
    return df, cluster_stats

In [ ]:
df_cluster, stats = query_spatial_clustering(df)

Statistiques par cluster spatial :
         nb_seismes  magnitude_moy  taux_tsunami
cluster                                         
0                88          6.920         0.511
1               115          6.937         0.530
2               238          6.933         0.277
3               165          6.939         0.224
4               133          6.974         0.519
5                43          6.949         0.605


INTERPRETATION

L'algorithme KMeans a identifié 6 zones géographiques différentes où les séismes se concentrent.

Analyse
Cluster 2 : zone la plus active
238 séismes, c'est le cluster qui contient le plus d'événements.
Magnitude moyenne de 6,93.
Seulement 27,7 % des séismes ont provoqué un tsunami.

 C'est la région la plus active sismiquement, mais les tsunamis y sont relativement moins fréquents.

Cluster 3 : deuxième zone la plus active
165 séismes.
Magnitude moyenne de 6,94.
Taux de tsunami de seulement 22,4 %.

 Beaucoup de séismes, mais peu de tsunamis.

Cluster 4 : séismes les plus puissants
Magnitude moyenne la plus élevée : 6,97.
133 séismes.
Plus de la moitié (51,9 %) ont généré un tsunami.

 Cette zone combine des séismes forts et un risque important de tsunami.

Cluster 5 : zone la plus dangereuse
Seulement 43 séismes.
Mais 60,5 % ont provoqué un tsunami.

 Même si cette zone connaît peu de séismes, elle présente le risque de tsunami le plus élevé.

In [ ]:
from dash import Dash, html, dcc
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc
import dash_bootstrap_components as dbc

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from mlxtend.frequent_patterns import apriori, association_rules

from pyngrok import ngrok


In [ ]:
from dash import Dash, html, dcc

In [ ]:
import plotly.express as px
import plotly.io as pio

In [ ]:
# ============================================================
# 1️ Magnitude maximale par année
# ============================================================
# On regroupe les données par année et on prend la magnitude max
mag_year = df.groupby('Year')['magnitude'].max().reset_index()

# Graphique en barres : évolution de la magnitude maximale par année
fig1 = px.bar(
    mag_year,
    x='Year',
    y='magnitude',
    title='Magnitude max par année',
    color='magnitude',
    color_continuous_scale='Reds'
)


# ============================================================
# 2️ Règles d’association (Apriori)
# ============================================================
# Si des règles existent, on les visualise
if rules is not None and not rules.empty:

    # Nuage de points des règles d’association
    fig2 = px.scatter(
        rules,
        x='support',                 # fréquence d’apparition
        y='confidence',              # fiabilité de la règle
        size='lift',                 # importance de la relation
        color='lift',
        title='Règles d’association (Apriori)',
        hover_data=['antecedents_str', 'consequents_str']  # détails visibles au survol
    )

# Sinon, on affiche un graphique vide avec message
else:
    fig2 = px.scatter(title="Aucune règle trouvée")


# ============================================================
# 3️ Analyse temporelle
# ============================================================
# Nombre de séismes par année
fig3 = px.bar(
    yearly,
    x='Year',
    y='count',
    title='Nombre de séismes par année',
    color='count',
    color_continuous_scale='Blues'
)


# ============================================================
# 4️ Analyse spatiale (clustering)
# ============================================================
# Visualisation géographique des séismes avec clusters
fig4 = px.scatter_geo(
    df_cluster,
    lat='latitude',              # position géographique
    lon='longitude',
    color='cluster',            # groupe de séismes
    size='magnitude',           # importance du séisme
    title='Clustering spatial des séismes',
    projection='natural earth'  # carte du monde
)


# ============================================================
# 5️ Fusion des graphiques en HTML
# ============================================================
# Liste des figures à afficher
figs = [fig1, fig2, fig3, fig4]

# Variable qui contiendra tout le HTML des graphiques
html_parts = ""

# Conversion de chaque graphique Plotly en HTML
for fig in figs:
    html_parts += fig.to_html(full_html=False, include_plotlyjs='cdn')


# ============================================================
# 6️ Création de la page HTML complète
# ============================================================
html_content = f"""
<html>
<head>
    <title>Earthquake Dashboard</title>
</head>

<body style="font-family: Arial;">

    <!-- Titre principal -->
    <h1 style="text-align:center;">
        Earthquake and Tsunami Analysis (2001–2022)
    </h1>

    <!-- Dataset utilisé -->
    <h3 style="text-align:center;">
        Dataset: earthquake_data_tsunami.csv
    </h3>

    <!-- Membres du projet -->
    <h3 style="text-align:center;">
        Team: Chanez Iglouli, Kamil Ghanam, Chahd Oulad-mhammed
    </h3>

    <hr>

    <!-- Insertion des graphiques -->
    {html_parts}

</body>
</html>
"""


# ============================================================
# 7️ Sauvegarde du dashboard en fichier HTML
# ============================================================
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# Message de confirmation
print("Dashboard HTML créé avec succès")

Dashboard HTML créé avec succès


In [ ]:






































































































































































































































from google.colab import files
files.download("dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:



# # ============================================================
# # Fonction principale : création du dashboard Dash
# # ============================================================
# def launch_dashboard(df, yearly, cluster_df, rules):

#     # Initialisation de l'application Dash avec un thème Bootstrap
#     app = Dash(
#         __name__,
#         external_stylesheets=[dbc.themes.BOOTSTRAP]
#     )

#     # =========================================================
#     # 1. Magnitude maximale par année
#     # =========================================================
#     # On calcule la plus grande magnitude enregistrée chaque année
#     mag_year = (
#         df.groupby('Year')['magnitude']
#         .max()
#         .reset_index()
#     )

#     # Graphique en barres pour visualiser les magnitudes maximales
#     fig1 = px.bar(
#         mag_year,
#         x='Year',
#         y='magnitude',
#         title='Maximum Magnitude per Year',
#         color='magnitude',
#         color_continuous_scale='Reds'
#     )

#     # =========================================================
#     # 2. Règles d’association (Apriori)
#     # =========================================================
#     # Si des règles existent, on les affiche
#     if rules is not None and not rules.empty:

#         fig2 = px.scatter(
#             rules,
#             x='support',
#             y='confidence',
#             size='lift',
#             color='lift',
#             title='Association Rules (Apriori)',
#             hover_data=[
#                 'antecedents_str',
#                 'consequents_str'
#             ]
#         )

#     # Sinon on affiche un graphique vide avec message
#     else:

#         fig2 = px.scatter(
#             title='No Association Rules Found'
#         )

#     # =========================================================
#     # 3. Évolution temporelle
#     # =========================================================
#     # Nombre de séismes par année
#     fig3 = px.bar(
#         yearly,
#         x='Year',
#         y='count',
#         title='Number of Earthquakes per Year',
#         color='count',
#         color_continuous_scale='Blues'
#     )

#     # =========================================================
#     # 4. Clustering spatial
#     # =========================================================
#     # Visualisation des clusters géographiques des séismes
#     fig4 = px.scatter_geo(
#         cluster_df,
#         lat='latitude',
#         lon='longitude',
#         color='cluster',
#         size='magnitude',
#         title='Spatial Clustering of Earthquakes',
#         projection='natural earth'
#     )

#     # =========================================================
#     # Mise en page du dashboard
#     # =========================================================
#     app.layout = html.Div([

#         # Titre principal
#         html.H1(
#             "Earthquake and Tsunami Analysis (2001–2022)",
#             style={'textAlign': 'center', 'color': '#2c3e50'}
#         ),

#         # Sous-titre dataset
#         html.H4(
#             "Dataset: earthquake_data_tsunami.csv",
#             style={'textAlign': 'center', 'color': 'gray'}
#         ),

#         # Membres de l'équipe
#         html.H5(
#             "Team members: Chanez Iglouli, Kamil Ghanam, Chahd Oulad-mhammed",
#             style={'textAlign': 'center', 'color': '#555'}
#         ),

#         html.Hr(),

#         # Première ligne de graphiques (2 colonnes)
#         dbc.Row([
#             dbc.Col(dcc.Graph(figure=fig1), width=6),
#             dbc.Col(dcc.Graph(figure=fig2), width=6)
#         ]),

#         # Deuxième ligne de graphiques (2 colonnes)
#         dbc.Row([
#             dbc.Col(dcc.Graph(figure=fig3), width=6),
#             dbc.Col(dcc.Graph(figure=fig4), width=6)
#         ]),

#         html.Br(),

#         # Footer du dashboard
#         html.Footer(
#             "Data Visualization and Large-scale Computing Project",
#             style={'textAlign': 'center', 'padding': '20px', 'color': 'gray'}
#         )

#     ])

#     # Message de confirmation
#     print("Dashboard créé avec succès")

#     return app

In [ ]:
# app = launch_dashboard(df, yearly, df_cluster, rules)

# port = 8050
# public_url = ngrok.connect(port)
# print(" Ouvre ce lien :", public_url)

# app.run_server(port=port, debug=False, use_reloader=False)